In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from matplotlib.ticker import FuncFormatter, PercentFormatter

# import auri
plt.style.use("./auri.mplstyle")
pd.options.display.unicode.east_asian_width = True

np.random.seed(1106)


In [37]:
df = pd.read_csv(
    "output/buildingstock/재고지수.csv",
    dtype={0: "object", "시군구명": "object"},
    # index_col=0
    # na_values=["-"],
)
df = df.set_index(df.columns[0])
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 250 entries, 11110 to 50130
Data columns (total 16 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   시군구명         250 non-null    object 
 1   연면적_합계       250 non-null    int64  
 2   주거용          250 non-null    int64  
 3   상업용          250 non-null    int64  
 4   공업용          250 non-null    int64  
 5   교육및사회용       250 non-null    int64  
 6   기타           250 non-null    int64  
 7   주민등록인구       250 non-null    int64  
 8   재고지표_주거용     250 non-null    float64
 9   재고지표_상업용     250 non-null    float64
 10  재고지표_공업용     250 non-null    float64
 11  재고지표_교육및사회용  250 non-null    float64
 12  재고지수_주거용     250 non-null    float64
 13  재고지수_상업용     250 non-null    float64
 14  재고지수_공업용     250 non-null    float64
 15  재고지수_교육및사회용  250 non-null    float64
dtypes: float64(8), int64(7), object(1)
memory usage: 33.2+ KB


In [36]:
df.index

Index(['11110', '11140', '11170', '11200', '11215', '11230', '11260', '11290',
       '11305', '11320',
       ...
       '48740', '48820', '48840', '48850', '48860', '48870', '48880', '48890',
       '50110', '50130'],
      dtype='object', name='Unnamed: 0', length=250)

In [41]:
# Get the numeric columns
numeric_columns = df.select_dtypes(include=["float64"]).columns

# Use pandas display options to format float values without altering the original data
with pd.option_context("display.float_format", "{:.1f}".format):
    # Sort and display top 10 and bottom 10 for each numeric column
    for col in numeric_columns:
        print(f"Top 10 sorted by {col}:")
        display(df[["시군구명", col]].sort_values(by=col, ascending=False).head(10))

        print(f"Bottom 10 sorted by {col}:")
        display(df[["시군구명", col]].sort_values(by=col, ascending=True).head(10))


Top 10 sorted by 재고지표_주거용:


,시군구명,재고지표_주거용
Unnamed: 0,,
42760,강원도 평창군,56.8
47900,경상북도 예천군,49.7
48840,경상남도 남해군,48.4
42830,강원도 양양군,47.7
46780,전라남도 보성군,47.6
43760,충청북도 괴산군,47.3
44270,충청남도 당진시,47.1
43800,충청북도 단양군,47.0
48720,경상남도 의령군,46.5


Bottom 10 sorted by 재고지표_주거용:


,시군구명,재고지표_주거용
Unnamed: 0,,
11620,서울특별시 관악구,25.8
41133,경기도 성남시 중원구,26.0
47940,경상북도 울릉군,26.9
46860,전라남도 함평군,27.2
42790,강원도 화천군,27.3
45750,전라북도 임실군,27.5
45790,전라북도 고창군,27.9
11545,서울특별시 금천구,28.5
11305,서울특별시 강북구,29.0


Top 10 sorted by 재고지표_상업용:


...

,시군구명,재고지수_공업용
Unnamed: 0,,
11380,서울특별시 은평구,0.1
11620,서울특별시 관악구,0.2
11590,서울특별시 동작구,0.2
11410,서울특별시 서대문구,0.3
11290,서울특별시 성북구,0.3
11650,서울특별시 서초구,0.3
11215,서울특별시 광진구,0.4
27200,대구광역시 남구,0.5
11305,서울특별시 강북구,0.9


Top 10 sorted by 재고지수_교육및사회용:


,시군구명,재고지수_교육및사회용
Unnamed: 0,,
11110,서울특별시 종로구,290.4
41820,경기도 가평군,264.3
30200,대전광역시 유성구,257.0
42760,강원도 평창군,242.4
11140,서울특별시 중구,226.0
45730,전라북도 무주군,209.6
29110,광주광역시 동구,208.2
44150,충청남도 공주시,207.1
42820,강원도 고성군,201.4


Bottom 10 sorted by 재고지수_교육및사회용:


,시군구명,재고지수_교육및사회용
Unnamed: 0,,
11260,서울특별시 중랑구,45.4
41450,경기도 하남시,51.2
41480,경기도 파주시,52.7
11320,서울특별시 도봉구,52.7
11380,서울특별시 은평구,54.1
11305,서울특별시 강북구,55.9
11545,서울특별시 금천구,55.9
11470,서울특별시 양천구,56.0
41570,경기도 김포시,56.7
